## יחס נסיגה ולולאת עדכון: מקרים שבהם לא ניתן לוקטר

לאורך כל הקורס עד כה, וקטוריזציה **תמיד** עבדה: כל איבר במערך התוצאה חושב מהנתונים המקוריים, בלי תלות באיברי תוצאה אחרים. הפעם יש מצב שבו זה נשבר: כאשר **צעד $n$ תלוי בתוצאה של צעד $n-1$** — יחס נסיגה. אז חייבים לולאת עדכון אמיתית.

### דוגמה: זריקה צעד אחרי צעד

עד עכשיו חישבנו את מסלול הזריקה מנוסחה סגורה. נבנה אותו הפעם **צעד אחרי צעד**: בכל צעד זמן ‎`dt`‎, המהירות והמיקום מתעדכנים לפי ערכם ב**צעד הקודם**:

$$v_{n+1} = v_n - g\,dt \qquad y_{n+1} = y_n + v_n\, dt$$

כך מחשבים תנועה כשאין לה נוסחה סגורה. כאן יש נוסחה, ולכן אפשר להשוות אליה את תוצאת הלולאה: בגרף שמתחת לתא, הלולאה בקו מלא והנוסחה בקו מקווקו, והן כמעט מתלכדות.

In [ ]:
import numpy as np

g = 9.8
theta = np.radians(45)
v0 = 20.0
dt = 0.01

n_steps = int(2 * v0 * np.sin(theta) / g / dt) + 1

y_sim = np.zeros(n_steps)
vy_sim = np.zeros(n_steps)
vy_sim[0] = v0 * np.sin(theta)

for n in range(n_steps - 1):
    vy_sim[n+1] = vy_sim[n] - g * dt          # needs the previous value
    y_sim[n+1] = y_sim[n] + vy_sim[n] * dt     # the same here

t_sim = np.arange(n_steps) * dt
y_exact = v0 * np.sin(theta) * t_sim - 0.5 * g * t_sim**2

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.plot(t_sim, y_sim, label="סימולציה (לולאה)")
ax.plot(t_sim, y_exact, color="gray", linestyle="--", label="נוסחה סגורה")
ax.set_xlabel("t (s)")
ax.set_ylabel("y (m)")
ax.legend()
plt.show()

### מעקב אחרי לולאה בטבלה

כשלולאת עדכון מדפיסה ערכים לא נכונים, הכלי הכי שימושי הוא טבלה: עמודה ל-‎`i`‎ ועמודה לכל מערך שהלולאה ממלאת. ממלאים אותה שורה אחרי שורה, כל שורה מתוך השורה שלפניה, בדיוק כמו המחשב. ניקח לולאה קטנה עם מספרים עגולים: זריקה אנכית במהירות 10 מטר לשנייה, עם ‎`g = 10`‎ ו-‎`dt = 0.1`‎, בארבעה צעדים.

In [ ]:
v = np.zeros(4)      # velocity (m/s)
y = np.zeros(4)      # height (m)
v[0] = 10.0
for i in range(3):
    v[i + 1] = v[i] - 10.0 * 0.1
    y[i + 1] = y[i] + v[i] * 0.1
print(v)
print(y)

| ‎`i`‎ | ‎`v[i]`‎ | ‎`y[i]`‎ | מה הסיבוב הזה מחשב |
|---|---|---|---|
| 0 | 10 | 0 | `v[1] = 10 - 1 = 9`, `y[1] = 0 + 10 * 0.1 = 1` |
| 1 | 9 | 1 | `v[2] = 9 - 1 = 8`, `y[2] = 1 + 9 * 0.1 = 1.9` |
| 2 | 8 | 1.9 | `v[3] = 8 - 1 = 7`, `y[3] = 1.9 + 8 * 0.1 = 2.7` |
| 3 | 7 | 2.7 | אין סיבוב כזה: ‎`range(3)`‎ נעצר לפני 3 |

כך משתמשים בטבלה כשהפלט שגוי:

1. ממלאים אותה לפי מה שהקוד **צריך** לעשות, עוד לפני שמסתכלים על הפלט.
2. משווים לפלט ומחפשים את הערך **הראשון** שיוצא שונה. כל ערך שאחריו כבר חושב מתוך ערך שגוי, ולכן הוא מלמד פחות.
3. השורה בקוד שמחשבת את הערך הזה היא החשודה הראשונה. אם ‎`v`‎ נכון ו-‎`y`‎ שגוי, הבעיה בשורה של ‎`y`‎. אם הלולאה נעצרת ב-‎`IndexError`‎, בודקים מה ה-‎`i`‎ האחרון ש-‎`range`‎ מייצר.

### באג נפוץ: ניסיון לוקטר יחס נסיגה

אפשר להתפתות לכתוב ‎`vy_sim[1:] = vy_sim[:-1] - g * dt`‎ בתקווה "לחסוך" את הלולאה — אבל זה **לא** עובד: הביטוי הזה מניח ש-‎`vy_sim[:-1]`‎ כבר מכיל את הערכים הנכונים, אבל בזמן ההרצה כל האיברים עדיין אפסים (חוץ מהראשון) — אין "שרשור" אמיתי בין הצעדים כשכותבים את זה כפעולת מערך אחת.

In [ ]:
vy_wrong = np.zeros(n_steps)
vy_wrong[0] = v0 * np.sin(theta)
vy_wrong[1:] = vy_wrong[:-1] - g * dt    # a wrong attempt to vectorize

print(vy_wrong[:5])   # from index 2 on, computed from the old zeros
print(vy_sim[:5])     # the right values, from the loop

ה"וקטוריזציה" השגויה מחשבת ‎`vy_wrong[:-1] - g*dt`‎ **פעם אחת**, מתוך המערך כפי שהיה *לפני* ההרצה — כמעט כולו אפסים חוץ מהאיבר הראשון. לכן רק ‎`vy_wrong[1]`‎ יוצא נכון (הוא מחושב מ-‎`vy_wrong[0]`‎, שכבר היה נכון), אבל ‎`vy_wrong[2]`‎ ואילך כולם מחושבים מ-**אפסים ישנים** ולא מהערכים המעודכנים בפועל — בדיוק כפי שרואים בהדפסה למעלה. ההבדל בין ‎`vy_wrong`‎ ל-‎`vy_sim`‎ הוא בדיוק ההבדל בין "כל האיברים בבת אחת, מהמצב הישן" לבין "כל איבר תלוי בערך המעודכן של קודמו".

בשפה של הטבלה: הערך הראשון ששונה הוא ‎`vy_wrong[2]`‎, וממנו והלאה כל הערכים שגויים.

### נסו בעצמכם

הוסיפו לטבלה שורה: אם הלולאה הקטנה הייתה רצה 5 צעדים (‎`np.zeros(5)`‎ ו-‎`range(4)`‎), מה היו ‎`v[4]`‎ ו-‎`y[4]`‎? מלאו את השורה לפני שאתם מריצים את התא.

In [ ]:
v = np.zeros(5)
y = np.zeros(5)
v[0] = 10.0
for i in range(4):
    v[i + 1] = v[i] - 10.0 * 0.1
    y[i + 1] = y[i] + v[i] * 0.1
print(v)
print(y)

`````{admonition} פתרון
:class: dropdown, tip
```text
[10.  9.  8.  7.  6.]
[0.  1.  1.9 2.7 3.4]
```
בסיבוב החדש, ‎`i = 3`‎:

- `v[4] = 7 - 1 = 6`
- `y[4] = 2.7 + 7 * 0.1 = 3.4`
`````

### בדקו את עצמכם

השאלה השנייה מתייחסת ללולאה הקטנה מהטבלה שלמעלה: ארבעה צעדים, ‎`g = 10`‎ ו-‎`dt = 0.1`‎.

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מדוע אי אפשר לחשב את כל המהירויות של לולאת העדכון בפעולת מערך אחת?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "כי NumPy לא תומך בחיסור",
                "correct": false,
                "feedback": "לא: חיסור נתמך היטב."
            },
            {
                "answer": "אפשר בהחלט, זו רק שאלה של סגנון",
                "correct": false,
                "feedback": "לא: ראינו שהתוצאה שגויה כשמנסים."
            },
            {
                "answer": "כי המהירות תמיד שלילית",
                "correct": false,
                "feedback": "לא: אין קשר בין סימן המהירות לבין האפשרות לחשב בפעולה אחת."
            },
            {
                "answer": "כי כל איבר תלוי בערך המעודכן של קודמו, וזה דורש לחשב צעד אחרי צעד",
                "correct": true,
                "feedback": "נכון."
            }
        ]
    },
    {
        "question": "בלולאה מהטבלה, מה הגובה באינדקס 2 של מערך הגבהים?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "1.9",
                "correct": true,
                "feedback": "נכון: הגובה הקודם, 1, ועוד המהירות הקודמת, 9, כפול 0.1."
            },
            {
                "answer": "1.8",
                "correct": false,
                "feedback": "לא: כאן חושב עם המהירות החדשה, 8. השורה של y משתמשת במהירות של הצעד הקודם."
            },
            {
                "answer": "0.9",
                "correct": false,
                "feedback": "לא: 0.9 הוא רק התוספת של הצעד הזה; צריך להוסיף אותה לגובה הקודם, 1."
            },
            {
                "answer": "2.0",
                "correct": false,
                "feedback": "לא: הגובה לא גדל באותה כמות בכל צעד, כי המהירות קטנה."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

### תרגול עצמי

עם התנגדות אוויר ליניארית פשוטה, המהירות האנכית מתעדכנת לפי $v_{n+1}=v_n - (g + k v_n)\,dt$. לזה **אין** נוסחה סגורה פשוטה, ולכן כאן לולאת עדכון היא לא רק נוחות אלא הכרח. השלימו את הפונקציה ‎`drag_velocities`‎ ובדקו אותה על הדוגמה שבתא. חשבו את שני הצעדים בטבלה לפני שאתם מריצים.

In [ ]:
def drag_velocities(v0, g, k, dt, n_steps):
    """Vertical velocity (m/s) at each step, with air resistance k (1/s)."""
    pass   # replace with your code

print(drag_velocities(10.0, 10.0, 1.0, 0.1, 3))
# expected: [10.   8.   6.2]

`````{admonition} פתרון
:class: dropdown, tip
```python
def drag_velocities(v0, g, k, dt, n_steps):
    """Vertical velocity (m/s) at each step, with air resistance k (1/s)."""
    v = np.zeros(n_steps)
    v[0] = v0
    for i in range(n_steps - 1):
        v[i + 1] = v[i] - (g + k * v[i]) * dt
    return v

print(drag_velocities(10.0, 10.0, 1.0, 0.1, 3))
```
- בסיבוב ‎`i = 0`‎: ‎`10 - (10 + 1 * 10) * 0.1 = 8`‎
- בסיבוב ‎`i = 1`‎: ‎`8 - (10 + 1 * 8) * 0.1 = 6.2`‎

בלי התנגדות (‎`k = 0`‎) היינו מקבלים 10, 9 ו-8: ההתנגדות מאטה את הגוף מהר יותר.
`````